## Ensemble: A5 + PubMedBERT-large

Combines two prediction files using three strategies. No retraining needed.

In [15]:
import json, re
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np

# ════════════════════════════════════════
# CONFIGURE
# ════════════════════════════════════════
PRED_A   = Path("predictions/inference_pubmedbert_large_re_A5_fullctx.json")   # micro F1 0.5899 — BEST A5
PRED_B   = Path("predictions/inference_pubmedbert_large_re_A5_hardneg.json")  # micro F1 0.5932
DEV_PATH = Path("../../data/GutBrainIE_Full_Collection_2026/Annotations/Dev/json_format/dev.json")
OUT_DIR  = Path("predictions")
# ════════════════════════════════════════

with open(PRED_A) as f: preds_a = json.load(f)
with open(PRED_B) as f: preds_b = json.load(f)
with open(DEV_PATH) as f: dev_data = json.load(f)

def norm(s): return re.sub(r"\s+", " ", str(s).strip())
def norm_ent(l): return "DDF" if str(l).strip().lower()=="ddf" else str(l).strip()

LEGAL_RELATION_LABELS = {
    "administered","affect","change abundance","change effect","change expression","compared to",
    "impact","influence","interact","is a","is linked to","located in","part of","produced by",
    "strike","target","used by"
}

def get_triples_set(preds, pmid):
    return {(norm(r["subject_text_span"]), norm_ent(r["subject_label"]),
             r["predicate"], norm(r["object_text_span"]), norm_ent(r["object_label"]))
            for r in preds.get(str(pmid), {}).get("mention_level_relations", [])
            if r["predicate"] in LEGAL_RELATION_LABELS}

def build_gold_maps(dev_data):
    gold = {}
    for pmid, art in dev_data.items():
        s = set()
        for r in art.get("mention_level_relations", []):
            if r["predicate"].strip() in LEGAL_RELATION_LABELS:
                s.add((norm(r["subject_text_span"]), norm_ent(r["subject_label"]),
                       r["predicate"].strip(), norm(r["object_text_span"]), norm_ent(r["object_label"])))
        gold[str(pmid)] = s
    return gold

def micro_scores(gold, pred):
    tp=fp=fn=0
    for pmid, g in gold.items():
        p = pred.get(pmid, set())
        tp+=len(g&p); fp+=len(p-g); fn+=len(g-p)
    P=tp/(tp+fp) if (tp+fp) else 0.0
    R=tp/(tp+fn) if (tp+fn) else 0.0
    F1=2*P*R/(P+R) if (P+R) else 0.0
    return {"P":round(P,4),"R":round(R,4),"F1":round(F1,4),"TP":tp,"FP":fp,"FN":fn}

def macro_scores(gold, pred):
    preds_list = sorted({t[2] for s in gold.values() for t in s})
    vals = []
    for pr in preds_list:
        tp=fp=fn=0
        for pmid, g in gold.items():
            gp={t for t in g if t[2]==pr}; pp={t for t in pred.get(pmid,set()) if t[2]==pr}
            tp+=len(gp&pp); fp+=len(pp-gp); fn+=len(gp-pp)
        P=tp/(tp+fp) if (tp+fp) else 0.0; R=tp/(tp+fn) if (tp+fn) else 0.0
        vals.append((P,R,2*P*R/(P+R) if (P+R) else 0.0))
    return {"macro_P":round(float(np.mean([x[0] for x in vals])),4),
            "macro_R":round(float(np.mean([x[1] for x in vals])),4),
            "macro_F1":round(float(np.mean([x[2] for x in vals])),4)}

gold_by_doc = build_gold_maps(dev_data)
pmids = list(dev_data.keys())

# Baseline
pred_a = {str(p): get_triples_set(preds_a, p) for p in pmids}
pred_b = {str(p): get_triples_set(preds_b, p) for p in pmids}

mi_a = micro_scores(gold_by_doc, pred_a)
mi_b = micro_scores(gold_by_doc, pred_b)
print(f"A5 base:          Micro F1={mi_a['F1']:.4f}  P={mi_a['P']:.4f}  R={mi_a['R']:.4f}")
print(f"PubMedBERT-large: Micro F1={mi_b['F1']:.4f}  P={mi_b['P']:.4f}  R={mi_b['R']:.4f}")


A5 base:          Micro F1=0.5872  P=0.5951  R=0.5795
PubMedBERT-large: Micro F1=0.5816  P=0.5541  R=0.6119


## Strategy 1 — Union

Keep a relation if **either** model predicts it. Maximum recall.

In [16]:
union_pred = {str(p): pred_a.get(str(p), set()) | pred_b.get(str(p), set()) for p in pmids}
mi_union = micro_scores(gold_by_doc, union_pred)
ma_union = macro_scores(gold_by_doc, union_pred)
print(f"Union:  Micro F1={mi_union['F1']:.4f}  P={mi_union['P']:.4f}  R={mi_union['R']:.4f}")
print(f"        Macro F1={ma_union['macro_F1']:.4f}")
print(f"  Delta vs A5: {mi_union['F1']-mi_a['F1']:+.4f}")


Union:  Micro F1=0.5784  P=0.5287  R=0.6383
        Macro F1=0.5319
  Delta vs A5: -0.0088


## Strategy 2 — Intersection

Keep a relation only if **both** models predict it. Maximum precision.

In [17]:
inter_pred = {str(p): pred_a.get(str(p), set()) & pred_b.get(str(p), set()) for p in pmids}
mi_inter = micro_scores(gold_by_doc, inter_pred)
ma_inter = macro_scores(gold_by_doc, inter_pred)
print(f"Intersection:  Micro F1={mi_inter['F1']:.4f}  P={mi_inter['P']:.4f}  R={mi_inter['R']:.4f}")
print(f"               Macro F1={ma_inter['macro_F1']:.4f}")
print(f"  Delta vs A5: {mi_inter['F1']-mi_a['F1']:+.4f}")


Intersection:  Micro F1=0.5913  P=0.6351  R=0.5531
               Macro F1=0.5470
  Delta vs A5: +0.0041


## Strategy 3 — Weighted Union by Predicate

For frequent predicates (many training examples): use intersection (high precision).
For rare predicates (few training examples): use union (recover missed relations).

In [18]:
# Predicati rari: pochi esempi gold → usa union per massimizzare recall
RARE = {"strike","change effect","produced by","compared to","change expression",
        "change abundance","part of","compared to"}
# Predicati frequenti: usa intersection per massimizzare precision
# FREQUENT = tutto il resto

weighted_pred = {}
for p in pmids:
    pmid = str(p)
    a = pred_a.get(pmid, set())
    b = pred_b.get(pmid, set())
    
    result = set()
    # Intersection per predicati frequenti
    result |= {t for t in (a & b)}
    # Union per predicati rari
    result |= {t for t in (a | b) if t[2] in RARE}
    
    weighted_pred[pmid] = result

mi_w = micro_scores(gold_by_doc, weighted_pred)
ma_w = macro_scores(gold_by_doc, weighted_pred)
print(f"Weighted:  Micro F1={mi_w['F1']:.4f}  P={mi_w['P']:.4f}  R={mi_w['R']:.4f}")
print(f"           Macro F1={ma_w['macro_F1']:.4f}")
print(f"  Delta vs A5: {mi_w['F1']-mi_a['F1']:+.4f}")


Weighted:  Micro F1=0.5863  P=0.6183  R=0.5575
           Macro F1=0.5311
  Delta vs A5: -0.0009


## Strategies D-G — Hybrid (from NER ensemble)

Base model + relations from extra model not already covered. Mirrors NER Strategy F.

In [19]:
# ── Strategie Hybrid ispirate al NER ensemble ──

# Strategy D — Hybrid A5-base: A5 come base + relazioni Large che A5 non ha
def build_hybrid(base_pred, extra_pred, pmids):
    """Base model + tutte le relazioni extra non già coperte dalla base."""
    result = {}
    for p in pmids:
        pmid = str(p)
        base = base_pred.get(pmid, set())
        extra = extra_pred.get(pmid, set())
        # chiave coppia (subj, subj_lab, obj, obj_lab) — ignora predicato
        covered_pairs = {(t[0], t[1], t[3], t[4]) for t in base}
        # aggiungi solo relazioni su coppie non ancora coperte
        new = {t for t in extra if (t[0], t[1], t[3], t[4]) not in covered_pairs}
        result[pmid] = base | new
    return result

# Strategy E — Hybrid Large-base: Large come base + relazioni A5 che Large non ha
def build_hybrid_rare(base_pred, extra_pred, pmids, rare_preds):
    """Base model + relazioni extra SOLO per predicati rari."""
    result = {}
    for p in pmids:
        pmid = str(p)
        base = base_pred.get(pmid, set())
        extra = extra_pred.get(pmid, set())
        covered_pairs = {(t[0], t[1], t[3], t[4]) for t in base}
        new = {t for t in extra 
               if t[2] in rare_preds 
               and (t[0], t[1], t[3], t[4]) not in covered_pairs}
        result[pmid] = base | new
    return result

RARE_PREDS = {"strike", "change effect", "produced by", "compared to",
              "change expression", "change abundance", "part of"}

hybrid_a5base  = build_hybrid(pred_a, pred_b, pmids)        # A5 base + Large extras
hybrid_lgbase  = build_hybrid(pred_b, pred_a, pmids)        # Large base + A5 extras
hybrid_rare_a5 = build_hybrid_rare(pred_a, pred_b, pmids, RARE_PREDS)  # A5 + Large su rari
hybrid_rare_lg = build_hybrid_rare(pred_b, pred_a, pmids, RARE_PREDS)  # Large + A5 su rari

for name, pred in [
    ("Hybrid A5-base+Large",       hybrid_a5base),
    ("Hybrid Large-base+A5",       hybrid_lgbase),
    ("Hybrid A5+Large(rare only)", hybrid_rare_a5),
    ("Hybrid Large+A5(rare only)", hybrid_rare_lg),
]:
    mi = micro_scores(gold_by_doc, pred)
    ma = macro_scores(gold_by_doc, pred)
    print(f"{name:<35} Micro F1={mi['F1']:.4f}  P={mi['P']:.4f}  R={mi['R']:.4f}  Macro={ma['macro_F1']:.4f}  vs A5: {mi['F1']-mi_a['F1']:+.4f}")


Hybrid A5-base+Large                Micro F1=0.5784  P=0.5287  R=0.6383  Macro=0.5319  vs A5: -0.0088
Hybrid Large-base+A5                Micro F1=0.5784  P=0.5287  R=0.6383  Macro=0.5319  vs A5: -0.0088
Hybrid A5+Large(rare only)          Micro F1=0.5853  P=0.5876  R=0.5830  Macro=0.5297  vs A5: -0.0019
Hybrid Large+A5(rare only)          Micro F1=0.5790  P=0.5487  R=0.6128  Macro=0.5333  vs A5: -0.0082


## Summary and Best Strategy

In [20]:
results = [
    ("A5 base",                   mi_a,     None),
    ("PubMedBERT-large",          mi_b,     None),
    ("Union",                     mi_union, union_pred),
    ("Intersection",              mi_inter, inter_pred),
    ("Weighted",                  mi_w,     weighted_pred),
    ("Hybrid A5-base+Large",      micro_scores(gold_by_doc, hybrid_a5base),  hybrid_a5base),
    ("Hybrid Large-base+A5",      micro_scores(gold_by_doc, hybrid_lgbase),  hybrid_lgbase),
    ("Hybrid A5+Large(rare)",     micro_scores(gold_by_doc, hybrid_rare_a5), hybrid_rare_a5),
    ("Hybrid Large+A5(rare)",     micro_scores(gold_by_doc, hybrid_rare_lg), hybrid_rare_lg),
]

print(f"{'Strategy':<22} {'Micro-F1':>9} {'Micro-P':>9} {'Micro-R':>9} {'vs A5':>8}")
print("-" * 62)
for name, mi, _ in results:
    delta = f"{mi['F1']-mi_a['F1']:+.4f}" if name != "A5 base" else "  base"
    print(f"{name:<22} {mi['F1']:>9.4f} {mi['P']:>9.4f} {mi['R']:>9.4f} {delta:>8}")

# Seleziona il migliore tra le strategie ensemble
best_name, best_mi, best_pred = max(
    [(n, m, p) for n, m, p in results if p is not None],
    key=lambda x: x[1]["F1"]
)
print(f"\n🏆 Best ensemble strategy: {best_name} (Micro F1={best_mi['F1']:.4f})")


Strategy                Micro-F1   Micro-P   Micro-R    vs A5
--------------------------------------------------------------
A5 base                   0.5872    0.5951    0.5795     base
PubMedBERT-large          0.5816    0.5541    0.6119  -0.0056
Union                     0.5784    0.5287    0.6383  -0.0088
Intersection              0.5913    0.6351    0.5531  +0.0041
Weighted                  0.5863    0.6183    0.5575  -0.0009
Hybrid A5-base+Large      0.5784    0.5287    0.6383  -0.0088
Hybrid Large-base+A5      0.5784    0.5287    0.6383  -0.0088
Hybrid A5+Large(rare)     0.5853    0.5876    0.5830  -0.0019
Hybrid Large+A5(rare)     0.5790    0.5487    0.6128  -0.0082

🏆 Best ensemble strategy: Intersection (Micro F1=0.5913)


## Save Best Ensemble Predictions

In [21]:
def triples_to_json(pred_by_doc, preds_source):
    """Convert triple sets back to JSON format."""
    out = {}
    for pmid in pmids:
        pmid_str = str(pmid)
        triples = pred_by_doc.get(pmid_str, set())
        relations = []
        for (st, sl, pred, ot, ol) in sorted(triples):
            relations.append({
                "subject_text_span": st,
                "subject_label": sl,
                "predicate": pred,
                "object_text_span": ot,
                "object_label": ol,
            })
        out[pmid_str] = {"mention_level_relations": relations}
    return out

OUT_DIR.mkdir(parents=True, exist_ok=True)

# Salva tutte e tre le strategie
for name, mi, pred in results:
    if pred is None:
        continue
    safe_name = name.lower().replace(" ", "_")
    out_path = OUT_DIR / f"ensemble_{safe_name}.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(triples_to_json(pred, None), f, ensure_ascii=False, indent=2)
    print(f"Saved: {out_path} (Micro F1={mi['F1']:.4f})")

print(f"\nBest: ensemble_{best_name.lower().replace(' ','_')}.json")


Saved: predictions\ensemble_union.json (Micro F1=0.5784)
Saved: predictions\ensemble_intersection.json (Micro F1=0.5913)
Saved: predictions\ensemble_weighted.json (Micro F1=0.5863)
Saved: predictions\ensemble_hybrid_a5-base+large.json (Micro F1=0.5784)
Saved: predictions\ensemble_hybrid_large-base+a5.json (Micro F1=0.5784)
Saved: predictions\ensemble_hybrid_a5+large(rare).json (Micro F1=0.5853)
Saved: predictions\ensemble_hybrid_large+a5(rare).json (Micro F1=0.5790)

Best: ensemble_intersection.json
